Import Dependencies

In [3]:
from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage,ToolMessage
from langchain_core.messages import convert_to_messages ,convert_to_openai_messages

from jinja2 import  Template
from typing import Literal,Dict,Any,Annotated,List
from IPython.display import Image,display
from operator import add
from openai import  OpenAI

import random
import ast
import inspect
import instructor
import json
from langchain_core.messages import AIMessage,ToolMessage,convert_to_openai_messages,HumanMessage,SystemMessage,BaseMessage
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct,Prefetch,FieldCondition,MatchText,FusionQuery,Document
import openai
import os
from langchain_openai import ChatOpenAI
from langsmith import traceable
from utils.tool import get_formatted_context,add_to_shopping_cart,remove_from_cart,get_shopping_cart,check_warehouse_availability,reserve_warehouse_items
from langgraph.checkpoint.postgres import PostgresSaver

from utils.utils import format_ai_message,parse_docstring_params,parse_function_definition

c:\Users\jaysi\Desktop\Desktop\Ai-engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c:\Users\jaysi\Desktop\Desktop\Ai-engineering\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [15]:
from litellm import completion

In [2]:
client=OpenAI()

In [10]:
class RAGUsedContext(BaseModel):
    id:str=Field(description="The ID Of the item used answer the questions")
    description:str=Field(description="Short description of the item used to answer the Question")


class Toolcall(BaseModel):
    name:str
    arguments:dict

class FinalResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

class AgentProperties(BaseModel):
   iteration:int=0
   available_tools:List[dict[str,Any]]=[]
   tool_calls:List[Toolcall]=[]
   final_answer:bool=False

class Delegation(BaseModel):
    agent:str
    task:str

class CoordinatorAgentProperties(BaseModel):
    iteration:int=0
    final_answer:bool=False
    plan:List[Delegation]=[]
    next_agent:str=""
    

class State(BaseModel):
    messages:Annotated[List[Any],add]=[]
    question_relevant:bool=False 
    user_intent:str=""
    answer:str=""
    product_qa_agent:AgentProperties=Field(default_factory=CoordinatorAgentProperties)
    references:Annotated[List[RAGUsedContext],add]=[]
    shopping_cart_agent:AgentProperties=Field(default_factory=CoordinatorAgentProperties)
    coordinator_agent:CoordinatorAgentProperties=Field(default_factory=CoordinatorAgentProperties)
    warehouse_manager_agent:AgentProperties=Field(default_factory=CoordinatorAgentProperties)
    user_id:str=""
    cart_id:str=""


In [5]:
class Delegation(BaseModel):
    agent:str
    task:str

class CoordinatorAgent(BaseModel):
    next_agent:str
    plan:List[Delegation]
    final_answer:bool=False
    answer:str

In [6]:
def getconvesationhistory(messages)->[List]:
    conversation=[]
    for message in messages:
        if type(message).__name__ == "HumanMessage":
            conversation.append(convert_to_openai_messages(message))
        elif isinstance(message, dict) and message.get("role") == "user":
            conversation.append(message)
        elif type(message).__name__ == "AIMessage":
            if message.content: 
                clean_msg = AIMessage(content=message.content)
                conversation.append(convert_to_openai_messages(clean_msg))
        elif isinstance(message, dict) and message.get("role") == "assistant":
            if message.get("content"):
                conversation.append({"role": "assistant", "content": message["content"]})
        elif type(message).__name__ == "ToolMessage":
            clean_msg = HumanMessage(content=f"Tool [{message.name}] Output:\n{message.content}")
            conversation.append(convert_to_openai_messages(clean_msg))
        elif isinstance(message, dict) and message.get("role") == "tool":
            conversation.append({"role": "user", "content": f"Tool [{message.get('name')}] Output:\n{message.get('content')}"})
        
    return conversation

In [16]:

@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={"ls_provider":"openai"}
)
def coordinator_agent(state):
    
    prompt_template="""  You are a Coordinator Agent as part of a shopping assistant.

    ## Instructions

    - Your role is to create plans for solving user queries and delegate the tasks accordingly.
    - You will be given a conversation history, your task is to create a plan for solving the user's query.
    - After the plan is created, you should output the next agent to invoke and the task to be performed by that agent.
    - Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and revise the plan.
    - If there is a sequence of tasks to be performed by a single agent, you should combine them into a single task.
    - Do not route to any agent if the user's query needs clarification or is irrelevant. Do it yourself.

    ## Available Agents

    - product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
    - shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
    - warehouse_manager_agent: The user is asking items from the Warehouse

    ## Examples

    Question: "Do you have running shoes under $100?"
    Next agent: product_qna_agent

    Question: "Can you list the items in my cart?"
    Next agent: shopping_cart_agent
    """
    template = Template(prompt_template)
    prompt = template.render()

    messages = state.messages
    conversation = []
    

    messages = state.messages
    conversation = getconvesationhistory(messages=messages)    
    client = instructor.from_litellm(completion)

    response,raw_response=client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        response_model=CoordinatorAgent,
        messages=[
            {
                "role":"system",

                 "content":prompt,
            },
            *conversation
        ],
        temperature=0.5,
    )

    if response.final_answer:
        ai_message=[AIMessage(content=response.answer)]
    else:
        ai_message=[]


    return {
        "messages":ai_message,
        "user_intent":"",
        "answer":response.answer,
        "coordinator_agent":{
            "iteration":state.coordinator_agent.iteration+1,
            "final_answer":response.final_answer,
            "next_agent":response.next_agent,
            "plan":[item.model_dump() for item in response.plan]

        }
    }



In [11]:
coordinator_agent(State(messages=[{"role":"user","content":"what is the Weeather Of Greater Noida"}]))

{'messages': [],
 'user_intent': '',
 'answer': 'I cannot provide weather information. You may want to check a weather-specific service or website for the current weather in Greater Noida.',
 'coordinator_agent': {'iteration': 1,
  'final_answer': False,
  'next_agent': '',
  'plan': []}}

### Coordinator Agent (LiteLLM Model Routing)

In [1]:
from litellm import completion

In [4]:
client = instructor.from_litellm(completion)

In [5]:
class SimpleResponce(BaseModel):
    answer:str

In [13]:
response,raw_response=client.chat.completions.create_with_completion(
        model="gpt-4.1",
        response_model=SimpleResponce,
        messages=[
            {
                "role":"system",

                 "content":"What Kind Of model family are you",
            },
            
        ],
        temperature=0.5,
    )

In [14]:
response.answer

'I am a large language model (LLM) based on the GPT-4 architecture, which is part of the transformer model family. Transformers are a type of neural network architecture that excels at understanding and generating human language. My training involved processing vast amounts of text data to learn patterns, context, and meaning, enabling me to assist with a wide range of tasks involving language.'